# Q3 baseline: the co-author seed and first journal entry

The question

- does having a co-author who already published in a journal raise the chance of entering that journal for the first time

The unit is one opportunity: one row per (author, journal, year) in which the author was active and had not previously published in the journal. `C` marks an earlier collaborator who had already published in that journal before year t. `F` marks the author's first entry in year t. The topic-adjusted comparison is the main Q3 analysis, so the notebook reports two stages

1. the naive contrast `F ~ C`, explicitly unadjusted
2. the adjustment `F ~ C + T` with Pierre's rolling topic fit, on the rows where `T` exists

The adjusted result carries stated limits. `T` exists on 17.2% of rows, so the complete cases are a selected population. The implementation uses complete cases as the explicit analysis population; missing T is not imputed. This is the fixed reporting position for Kevin’s part and a proposal for the joint report until the team agrees. The frozen-profile sensitivity has no common anchor for `C = 0`. The adjusted intervals are author-clustered delta-method intervals, not the author bootstrap named in the original plan. A separate Prolog implementation matched `C`, `F`, and the ride flag on all 6,422,558 rows. Variant A is the main opportunity set and B the sensitivity, so this notebook uses variant A only.

Inputs fetched and hash-checked from the configured shared folder; paths below are relative to this notebook

- `event_table_python_v0_oppA.csv`, 6,422,558 rows
- `event_table_topicmatch_v5.csv`, the same rows with `topic_match` added, Pierre's official v5 export (sha256 8a9e8a92, 497,765,187 bytes): threshold removed, journal profiles from the full corpus, adapter active. An independent local regeneration of the same pipeline matches it on every status and count column and on all 1,106,356 topic values to 4.7e-7.
- `event_table_topicmatch_local_min3.csv`, the superseded thresholded run, read only in step 8 as the historical comparison


## Step 1: load and align

- both files carry the same rows in the same order, which is asserted on the key columns before the topic column is attached
- `F` is read off `entering_work_id`, filled exactly on entry rows

In [1]:
# Fetch verified shared inputs; the existing analysis paths stay the same.
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parent
if not (project_root / "data-manifest.json").is_file():
    raise RuntimeError("Run this notebook with its working directory set to B_opportunities_and_analysis/.")
sys.path.insert(0, str(project_root))
from project_data import ensure_data
ensure_data("q3", root=project_root)

import numpy as np, pandas as pd

usecols = ["author_id", "journal_id", "t", "n_prior_papers", "coauthor_seed",
           "first_entry_ride", "entering_work_id"]
ev = pd.read_csv("../B_opportunities_and_analysis/data/event_table_python_v0_oppA.csv", usecols=usecols, low_memory=False,
                 dtype={"t": "int16", "n_prior_papers": "int32", "coauthor_seed": "int8",
                        "first_entry_ride": "int8", "entering_work_id": "str"})
tm = pd.read_csv("../C_topic_match/data/event_table_topicmatch_v5.csv", low_memory=False,
                 usecols=["author_id", "journal_id", "t", "topic_match", "tm_status", "profile_cutoff"],
                 dtype={"t": "int16"})

# same rows in the same order, otherwise attaching by position would silently mix rows
assert len(ev) == len(tm)
assert (ev["author_id"].values == tm["author_id"].values).all()
assert (ev["journal_id"].values == tm["journal_id"].values).all()
assert (ev["t"].values == tm["t"].values).all()
ev["T"] = tm["topic_match"].values
ev["tm_status"] = tm["tm_status"].values

# the status column must explain the fill exactly, and every profile must stop before t
assert ((tm["tm_status"] == "ok") == tm["topic_match"].notna()).all()
assert (tm.loc[tm.tm_status == "ok", "profile_cutoff"] < tm.loc[tm.tm_status == "ok", "t"]).all()
del tm

ev["F"] = ev["entering_work_id"].notna().astype("int8")
C = ev["coauthor_seed"].values
F = ev["F"].values
print(f"{len(ev):,} rows, {int(F.sum()):,} entries, {(C==1).sum():,} rows with a seed")

verified B_opportunities_and_analysis/data/event_table_python_v0_oppA.csv


verified C_topic_match/data/event_table_topicmatch_v5.csv


verified C_topic_match/data/event_table_topicmatch_local_min3.csv


6,422,558 rows, 96,819 entries, 19,035 rows with a seed


## Step 2: check against the build

The event table build printed its own headline counts. Rebuilding them here from the loaded frame confirms nothing was lost or doubled on the way in.

In [2]:
assert int(F.sum()) == 96819                       # one entry row per (author, journal) pair in the corpus
assert int((C == 1).sum()) == 19035                # rows where the seed predates t
assert int(F[C == 1].sum()) == 1784                # entries that happened with a seed in place
assert int(ev["first_entry_ride"].sum()) == 756    # entries with the qualifying seed co-author on the paper
# T coverage changes between export versions, the fill contract is asserted at load time instead
print(f"all four event counts match the build, T exists on {ev['T'].notna().sum():,} rows in this export")

all four event counts match the build, T exists on 1,106,356 rows in this export


## Step 3: the naive contrast

- entry rate with a seed against entry rate without, nothing held constant
- this number is confounded by design, an author with a seed is better connected and probably closer to the journal's topics, so read it as a crude association, not an effect in either direction
- the interval resamples whole authors, because one author contributes many related rows and row-level intervals would be too narrow
- two ratios are reported, as documented in the August entry: Q3_all counts every first entry, Q3_ind counts only entries without the seed co-author on the entering paper, each against the unseeded entry rate

In [3]:
p1, p0 = F[C == 1].mean(), F[C == 0].mean()
print(f"entry rate with a seed    {p1:.4%}  ({int(F[C==1].sum()):,} of {(C==1).sum():,})")
print(f"entry rate without        {p0:.4%}  ({int(F[C==0].sum()):,} of {(C==0).sum():,})")
print(f"naive rate ratio          {p1/p0:.2f}")

# author level counts once, then the bootstrap only touches these four arrays
codes, authors = pd.factorize(ev["author_id"])
nA = len(authors)
n1 = np.bincount(codes[C == 1], minlength=nA)
e1 = np.bincount(codes[(C == 1) & (F == 1)], minlength=nA)
n0 = np.bincount(codes[C == 0], minlength=nA)
e0 = np.bincount(codes[(C == 0) & (F == 1)], minlength=nA)

rng = np.random.default_rng(31)
reps = []
for _ in range(2000):
    idx = rng.integers(0, nA, nA)   # draw authors with replacement, rows follow their author
    a, b, c, d = e1[idx].sum(), n1[idx].sum(), e0[idx].sum(), n0[idx].sum()
    if b and c and d:
        reps.append((a / b) / (c / d))
lo, hi = np.percentile(reps, [2.5, 97.5])
print(f"author-clustered 95% CI   [{lo:.2f}, {hi:.2f}]  ({len(reps)} bootstrap draws)")

# Q3_ind, the headline documented in the August entry: entries without the seed co-author on the paper over the unseeded entry rate
ride_flag = ev["first_entry_ride"].values
Find = ((F == 1) & (ride_flag == 0)).astype("int8")
print(f"\nQ3_ind crude              {Find[C == 1].mean() / F[C == 0].mean():.2f}  "
      f"({int(Find[C == 1].sum()):,} independent entries on {(C == 1).sum():,} seeded rows)")
i1 = np.bincount(codes[(C == 1) & (Find == 1)], minlength=nA)
rng_ind = np.random.default_rng(32)
reps_ind = []
for _ in range(2000):
    idx = rng_ind.integers(0, nA, nA)
    a, b, c_, d = i1[idx].sum(), n1[idx].sum(), e0[idx].sum(), n0[idx].sum()
    if b and c_ and d:
        reps_ind.append((a / b) / (c_ / d))
lo_i, hi_i = np.percentile(reps_ind, [2.5, 97.5])
print(f"author-clustered 95% CI   [{lo_i:.2f}, {hi_i:.2f}]")

entry rate with a seed    9.3722%  (1,784 of 19,035)
entry rate without        1.4841%  (95,035 of 6,403,523)
naive rate ratio          6.32


author-clustered 95% CI   [6.04, 6.59]  (2000 bootstrap draws)

Q3_ind crude              3.64  (1,028 independent entries on 19,035 seeded rows)


author-clustered 95% CI   [3.42, 3.85]


## Step 4: ride against independent, inside the seeded entries

Of the entries that happened with a seed in place, some carry the seed co-author on the entering paper itself and some do not. The split matters because riding along and entering independently are different mechanisms, and only the seeded entries can show it.

In [4]:
rides = int(ev.loc[(C == 1) & (F == 1), "first_entry_ride"].sum())
seeded_entries = int(F[C == 1].sum())
print(f"seeded entries {seeded_entries:,}")
print(f"  with the seed co-author on the entering paper   {rides:,}  ({rides/seeded_entries:.0%})")
print(f"  without the seed co-author on the paper         {seeded_entries-rides:,}  ({1-rides/seeded_entries:.0%})")

seeded entries 1,784
  with the seed co-author on the entering paper   756  (42%)
  without the seed co-author on the paper         1,028  (58%)


## Step 5: the contrast by year

The corpus starts in 2015, so no observed co-author relation can predate that year. Seeded cells only reach a useful size around 2021. Earlier estimates are based on very few entries and should not be read as a trend.

In [5]:
g = ev.groupby("t", observed=True)
yr = pd.DataFrame({
    "seeded_rows": g.apply(lambda x: int((x.coauthor_seed == 1).sum()), include_groups=False),
    "seeded_entries": g.apply(lambda x: int(((x.coauthor_seed == 1) & (x.F == 1)).sum()), include_groups=False),
    "other_rows": g.apply(lambda x: int((x.coauthor_seed == 0).sum()), include_groups=False),
    "other_entries": g.apply(lambda x: int(((x.coauthor_seed == 0) & (x.F == 1)).sum()), include_groups=False),
})
yr["rate_ratio"] = (yr.seeded_entries / yr.seeded_rows) / (yr.other_entries / yr.other_rows)
print(yr.to_string(float_format=lambda v: f"{v:.2f}"))
print("\n2016 has 5 seeded entries; later years have more. Year-specific ratios are descriptive, not evidence of a time trend.")

      seeded_rows  seeded_entries  other_rows  other_entries  rate_ratio
t                                                                       
2015            0               0      117056           1855         NaN
2016            9               5      161085           2461       36.36
2017           46              12      196118           2952       17.33
2018          185              39      326591           4906       14.03
2019          512              77      558551           8475        9.91
2020         1001             141      723699          10631        9.59
2021         1739             170      856449          12734        6.57
2022         3314             343     1052619          15723        6.93
2023         5243             441     1122706          16497        5.72
2024         6986             556     1288649          18801        5.46

2016 has 5 seeded entries; later years have more. Year-specific ratios are descriptive, not evidence of a time trend.


## Step 6: what T covers, with the reasons named

`topic_match` is the rolling author profile against the journal profile, 0 to 1, from Pierre's v5 export. `profile_cutoff` sits strictly before t on every filled row, and every empty cell names its reason in `tm_status`

- the whole-period paper threshold is removed, so eligibility no longer looks at future productivity, and journal profiles now build from every embeddable paper
- the remaining gaps are structural, most entrants have no pre-t paper at all, and a smaller block has papers without usable abstracts or journals without a profile before t

In [6]:
has = ev["T"].notna().values
print(f"T exists on {has.sum():,} rows ({has.mean():.1%})")
print(f"  on entry rows            {has[F==1].mean():.1%}")
print(f"  on seeded rows           {has[C==1].mean():.1%}")
print(f"  on seeded entries        {has[(C==1)&(F==1)].mean():.1%}  ({int(has[(C==1)&(F==1)].sum()):,} of {int(((C==1)&(F==1)).sum()):,})")
print(f"  on rides                 {has[ev.first_entry_ride.values==1].mean():.1%}")
print("\nwhy the rest is empty, all rows")
print(ev["tm_status"].value_counts().to_string())
print("\non entry rows")
print(ev.loc[F == 1, "tm_status"].value_counts().to_string())
print("\non seeded entries")
print(ev.loc[(F == 1) & (C == 1), "tm_status"].value_counts().to_string())

T exists on 1,106,356 rows (17.2%)
  on entry rows            12.2%
  on seeded rows           99.2%
  on seeded entries        99.0%  (1,766 of 1,784)
  on rides                 99.6%

why the rest is empty, all rows


tm_status
no_author_history                4989914
ok                               1106356
no_author_and_journal_history     253203
below_threshold_no_abstract        55822
no_journal_history                 17263

on entry rows
tm_status
no_author_history                82281
ok                               11779
no_author_and_journal_history     1931
below_threshold_no_abstract        823
no_journal_history                   5

on seeded entries
tm_status
ok                   1766
no_author_history      18


## Step 7: adjustment, F ~ C + T on the rows where T exists

The rolling topic fit may partly capture a mediator, and complete cases form a selected subset of the full table, so these estimates describe the measurable-T population, not the whole table. The reported models concern the measurable-T population and carry the coverage limitation; no full-population adjusted claim is made.

- logistic regression fitted by Newton's method, a fit that does not converge stops the notebook instead of returning a number
- standard errors clustered by author
- the adjusted ratio comes from predicting every included row once with `C = 1` and once with `C = 0`, then dividing the two mean predicted risks, with a delta-method interval from the clustered variance
- overlap is checked first, adjustment only means something where seeded and unseeded rows share the same range of `T`


In [7]:
m = has
X = np.column_stack([np.ones(m.sum()), C[m].astype(float), ev["T"].values[m]])
y = F[m].astype(float)
print(f"complete cases {int(m.sum()):,} rows, {int(y.sum()):,} entries")
print(f"naive ratio inside this population {(y[X[:,1]==1].mean())/(y[X[:,1]==0].mean()):.2f}, "
      f"against 6.32 in the full table, so the complete cases are a selected population\n")

# overlap in T between the two exposure groups
q = [0.05, 0.25, 0.5, 0.75, 0.95]
qt1 = np.quantile(X[X[:, 1] == 1, 2], q)
qt0 = np.quantile(X[X[:, 1] == 0, 2], q)
print("T quantiles 5/25/50/75/95")
print("  with seed    " + "  ".join(f"{v:.2f}" for v in qt1))
print("  without      " + "  ".join(f"{v:.2f}" for v in qt0))
print()

def newton_logit(X, y, max_iter=50, tol=1e-10):
    # maximum likelihood logit by newton's method, every fit in this notebook goes through here
    # a fit that runs out of iterations, leaves the finite range or ends with a non-zero score raises,
    # so a non-converged fit (for example under complete separation) can never print as a result
    b = np.zeros(X.shape[1])
    for it in range(max_iter):
        p = 1 / (1 + np.exp(-(X @ b)))
        score = X.T @ (y - p)
        H = (X * (p * (1 - p))[:, None]).T @ X
        try:
            step = np.linalg.solve(H, score)
        except np.linalg.LinAlgError:
            raise RuntimeError(f"logit fit hit a singular hessian at iteration {it}, separated or redundant design")
        b += step
        if not np.isfinite(b).all():
            raise RuntimeError(f"logit fit left the finite range at iteration {it}")
        if np.abs(step).max() < tol:
            break
    else:
        raise RuntimeError(f"logit fit did not converge in {max_iter} iterations, last step {np.abs(step).max():.3g}")
    p = 1 / (1 + np.exp(-(X @ b)))
    score = X.T @ (y - p)
    if not np.isfinite(score).all() or np.abs(score).max() > 1e-6 * len(y):
        raise RuntimeError(f"logit fit ended with a non-zero score {np.abs(score).max():.3g}")
    return b

beta = newton_logit(X, y)

# sandwich variance with author clusters
p = 1 / (1 + np.exp(-(X @ beta)))
cl = codes[m]
U = X * (y - p)[:, None]
order = np.argsort(cl)
S = np.zeros((3, 3))
for blk in np.split(U[order], np.flatnonzero(np.diff(cl[order])) + 1):
    s = blk.sum(0)
    S += np.outer(s, s)
Hinv = np.linalg.inv((X * (p * (1 - p))[:, None]).T @ X)
se = np.sqrt(np.diag(Hinv @ S @ Hinv))

print(f"C   log odds {beta[1]:.3f}, OR {np.exp(beta[1]):.2f}, "
      f"cluster 95% CI [{np.exp(beta[1]-1.96*se[1]):.2f}, {np.exp(beta[1]+1.96*se[1]):.2f}]")
print(f"T   log odds {beta[2]:.3f}, per 0.1 of topic fit OR {np.exp(beta[2]/10):.2f}")

X1 = np.column_stack([X[:, 0], np.ones(len(X)), X[:, 2]])
X0 = np.column_stack([X[:, 0], np.zeros(len(X)), X[:, 2]])

def log_rr(b):
    p1 = 1 / (1 + np.exp(-(X1 @ b)))
    p0 = 1 / (1 + np.exp(-(X0 @ b)))
    return np.log(p1.mean() / p0.mean())

# delta method, the ratio is a smooth function of beta so its variance follows from the sandwich
grad = np.zeros(3)
for k in range(3):
    d = np.zeros(3); d[k] = 1e-6
    grad[k] = (log_rr(beta + d) - log_rr(beta - d)) / 2e-6
se_lrr = float(np.sqrt(grad @ (Hinv @ S @ Hinv) @ grad))
rr = np.exp(log_rr(beta))
print(f"\nadjusted rate ratio, g computation  {rr:.2f}, "
      f"cluster 95% CI [{rr*np.exp(-1.96*se_lrr):.2f}, {rr*np.exp(1.96*se_lrr):.2f}]")

# Q3_ind adjusted, the outcome is entry without the seed co-author on the paper, under C = 0 that is every entry
yind = ((F[m] == 1) & (ev["first_entry_ride"].values[m] == 0)).astype(float)
bi = newton_logit(X, yind)
pi_ = 1 / (1 + np.exp(-(X @ bi)))
Ui = X * (yind - pi_)[:, None]
Si = np.zeros((3, 3))
for blk in np.split(Ui[order], np.flatnonzero(np.diff(cl[order])) + 1):
    s_ = blk.sum(0)
    Si += np.outer(s_, s_)
Hi = np.linalg.inv((X * (pi_ * (1 - pi_))[:, None]).T @ X)

def log_rr_ind(b):
    return np.log((1 / (1 + np.exp(-(X1 @ b)))).mean() / (1 / (1 + np.exp(-(X0 @ b)))).mean())

gi = np.zeros(3)
for k in range(3):
    d = np.zeros(3); d[k] = 1e-6
    gi[k] = (log_rr_ind(bi + d) - log_rr_ind(bi - d)) / 2e-6
se_i = float(np.sqrt(gi @ (Hi @ Si @ Hi) @ gi))
rri = np.exp(log_rr_ind(bi))
print(f"Q3_ind adjusted, g computation      {rri:.2f}, cluster 95% CI [{rri*np.exp(-1.96*se_i):.2f}, {rri*np.exp(1.96*se_i):.2f}]")

complete cases 1,106,356 rows, 11,779 entries
naive ratio inside this population 10.16, against 6.32 in the full table, so the complete cases are a selected population

T quantiles 5/25/50/75/95
  with seed    0.74  0.79  0.83  0.88  0.93
  without      0.66  0.72  0.76  0.81  0.88



C   log odds 1.875, OR 6.52, cluster 95% CI [6.14, 6.93]
T   log odds 8.817, per 0.1 of topic fit OR 2.41

adjusted rate ratio, g computation  6.09, cluster 95% CI [5.76, 6.45]


Q3_ind adjusted, g computation      3.34, cluster 95% CI [3.12, 3.58]


## Step 8: where the change in the adjusted ratio comes from

The thresholded and threshold-free runs do not measure exactly the same `T`: the journal profiles change with the included papers, and the kernel scale is re-estimated. The same model is therefore run three times: first with the old values on the shared rows, then with the new values on those rows, and finally with all newly covered rows. This is a diagnostic sequence, not a unique attribution, because its order affects the intermediate result.

In [8]:
# the thresholded run, values on the same rows for the value-swap comparison
tmin3 = pd.read_csv("../C_topic_match/data/event_table_topicmatch_local_min3.csv", low_memory=False,
                    usecols=["author_id", "journal_id", "t", "topic_match"], dtype={"t": "int16"})
assert (tmin3["author_id"].values == ev["author_id"].values).all()   # same rows in the same order
assert (tmin3["journal_id"].values == ev["journal_id"].values).all()
assert (tmin3["t"].values == ev["t"].values).all()
told = tmin3["topic_match"].values
del tmin3

def adjusted_rr(mask, T):
    Xd = np.column_stack([np.ones(mask.sum()), C[mask].astype(float), T[mask]])
    yd = F[mask].astype(float)
    b = newton_logit(Xd, yd)
    p1 = (1 / (1 + np.exp(-(np.column_stack([Xd[:, 0], np.ones(len(Xd)), Xd[:, 2]]) @ b)))).mean()
    p0 = (1 / (1 + np.exp(-(np.column_stack([Xd[:, 0], np.zeros(len(Xd)), Xd[:, 2]]) @ b)))).mean()
    return p1 / p0

tnew = ev["T"].values
shared = ~np.isnan(told) & ~np.isnan(tnew)
print("kernel scale, thresholded run 0.3613 against v5 0.3635, both from run metadata, so nearly unchanged")
print(f"shared rows {shared.sum():,}")
print(f"  old values, shared rows        {adjusted_rr(shared, told):.2f}")
print(f"  new values, same shared rows   {adjusted_rr(shared, tnew):.2f}")
print(f"  new values, full new rows      {adjusted_rr(~np.isnan(tnew), tnew):.2f}")

kernel scale, thresholded run 0.3613 against v5 0.3635, both from run metadata, so nearly unchanged
shared rows 623,510


  old values, shared rows        5.24


  new values, same shared rows   5.69


  new values, full new rows      6.09


## Step 9: journal, year and prior paper count

The C + T model does not account for differences in entry rates between journals or years. We compare it with separate additive journal and year terms. Both outcomes and all specifications use the same rows: we exclude the union of journals with zero events for either outcome and check that each remaining journal and year has both events and non-events. This addresses those simple forms of separation, not every possible separated design. The exclusion depends on the outcomes and limits the population.

We also add `log1p(n_prior_papers)` to each specification. This is a new sensitivity analysis: more prior papers may relate to both connection and entry, and the size of a topic profile can affect T. The log form allows a larger difference between one and two papers than between fifty and fifty-one; it is a modelling choice, not a validated functional form. We chose it before running this extension. The count covers earlier papers in our corpus, not just the embedded papers used in the profile (`n_profile_papers`). It can itself reflect earlier collaboration, so adding it does not establish a causally sufficient adjustment set or separate topic from productivity.

Every ratio below has a nominal 95% author-clustered delta interval, using the same sandwich convention as Step 7 (no small-sample correction). These describe sampling uncertainty under the fitted model and independent author clusters. Shared-paper dependence across authors, model choice and selection from missing T are not covered. We keep the original reporting comparison alongside these additional specifications; no team approval or change to the headline is implied.


In [9]:
# Use the same rows for both outcomes and all four specifications.
mfe = m.copy()
outcomes = pd.DataFrame({
    "Q3_all": F[m],
    "Q3_ind": ((F[m] == 1) & (ev["first_entry_ride"].values[m] == 0)).astype(int),
    "journal_id": ev.loc[m, "journal_id"].values,
})
jent = outcomes.groupby("journal_id", observed=True)[["Q3_all", "Q3_ind"]].sum()
zero_by_outcome = {name: jent.index[jent[name].eq(0)].tolist()
                   for name in ("Q3_all", "Q3_ind")}
for name, journals in zero_by_outcome.items():
    print(f"{name}: journals without this outcome: {journals}")
dropped = sorted(set().union(*zero_by_outcome.values()))
mfe &= ~ev["journal_id"].isin(dropped).values
print(f"common exclusion: {dropped}; {int(m.sum() - mfe.sum()):,} rows dropped")
if not mfe.any():
    raise RuntimeError("No rows remain after the common outcome-based exclusion")
jc, jl = pd.factorize(ev.loc[mfe, "journal_id"])
yc, yl = pd.factorize(ev.loc[mfe, "t"])
cl_fe, author_levels = pd.factorize(ev.loc[mfe, "author_id"])
yall = F[mfe].astype(float)
yind = ((F[mfe] == 1) & (ev["first_entry_ride"].values[mfe] == 0)).astype(float)
# A converged optimizer is not a substitute for checking each outcome's support.
for name, yv in (("Q3_all", yall), ("Q3_ind", yind)):
    for field in ("journal_id", "t"):
        support = pd.DataFrame({"group": ev.loc[mfe, field].values, "y": yv}).groupby("group")["y"].agg(["sum", "count"])
        bad = support.index[(support["sum"] == 0) | (support["sum"] == support["count"])].tolist()
        if bad:
            raise RuntimeError(f"{name}: {field} groups without outcome variation: {bad}; review the common population before fitting")
print(f"{int(mfe.sum()):,} rows, {len(jl)} journals, {len(yl)} years, {len(author_levels):,} authors")

prior = ev["n_prior_papers"].values[mfe].astype(float)
assert np.isfinite(prior).all() and (prior >= 0).all()
print("prior corpus papers, min/median/90%/max:", np.quantile(prior, [0, .5, .9, 1]))
log_prior = np.log1p(prior)

def design_fe(cvals):
    n = len(cvals)
    Xf = np.zeros((n, 3 + len(jl) - 1 + len(yl) - 1), dtype=float)
    Xf[:, 0] = 1; Xf[:, 1] = cvals; Xf[:, 2] = ev["T"].values[mfe]
    for j in range(1, len(jl)): Xf[jc == j, 2 + j] = 1
    for year in range(1, len(yl)): Xf[yc == year, 1 + len(jl) + year] = 1
    return Xf

def standardized_rr_ci(Xf, yv, clusters):
    """Standardized RR and nominal author-clustered delta interval (no small-sample correction)."""
    b = newton_logit(Xf, yv, tol=1e-8)
    n, k = Xf.shape
    _, cl = np.unique(clusters, return_inverse=True)
    if cl.max() < 1:
        raise ValueError("At least two author clusters are needed")
    scores = np.zeros((cl.max() + 1, k))
    H = np.zeros((k, k))
    risk1 = risk0 = 0.0
    derivative1 = np.zeros(k); derivative0 = np.zeros(k)
    # Blocks keep the covariance and prediction work below another full design copy.
    for start in range(0, n, 25000):
        stop = min(start + 25000, n)
        xb = Xf[start:stop]
        p = 1 / (1 + np.exp(-(xb @ b)))
        H += (xb * (p * (1 - p))[:, None]).T @ xb
        u = xb * (yv[start:stop] - p)[:, None]
        for j in range(k):
            scores[:, j] += np.bincount(cl[start:stop], weights=u[:, j], minlength=len(scores))
        xcf = xb.copy(); xcf[:, 1] = 1
        p1 = 1 / (1 + np.exp(-(xcf @ b)))
        risk1 += p1.sum(); derivative1 += xcf.T @ (p1 * (1 - p1))
        xcf[:, 1] = 0
        p0 = 1 / (1 + np.exp(-(xcf @ b)))
        risk0 += p0.sum(); derivative0 += xcf.T @ (p0 * (1 - p0))
    Hinv = np.linalg.inv(H)
    covariance = Hinv @ (scores.T @ scores) @ Hinv
    gradient = derivative1 / risk1 - derivative0 / risk0
    variance = float(gradient @ covariance @ gradient)
    if not np.isfinite(variance) or variance < 0 or min(risk0, risk1) <= 0:
        raise RuntimeError("Invalid variance or standardized risks; do not report this fit")
    rr = risk1 / risk0
    se_log_rr = np.sqrt(variance)
    return {"rr": rr, "lo": rr * np.exp(-1.96 * se_log_rr),
            "hi": rr * np.exp(1.96 * se_log_rr), "se_log_rr": se_log_rr}

Xfe = design_fe(C[mfe].astype(float))
fe_results = []
for specification, use_fe, use_prior in (
    ("C + T", False, False),
    ("C + T + log1p(prior papers)", False, True),
    ("C + T + journal + year", True, False),
    ("C + T + journal + year + log1p(prior papers)", True, True),
):
    design = Xfe if use_fe else Xfe[:, :3]
    if use_prior:
        design = np.column_stack([design, log_prior])
    for name, yv in (("Q3_all", yall), ("Q3_ind", yind)):
        result = standardized_rr_ci(design, yv, cl_fe)
        fe_results.append({"outcome": name, "specification": specification, **result})
        print(f"{name} | {specification}: {result['rr']:.4f} [{result['lo']:.4f}, {result['hi']:.4f}]", flush=True)
    del design
fe_results = pd.DataFrame(fe_results)
del Xfe


Q3_all: journals without this outcome: ['S4210228265']
Q3_ind: journals without this outcome: ['S4210228265']


common exclusion: ['S4210228265']; 17,936 rows dropped


1,088,420 rows, 63 journals, 9 years, 12,529 authors
prior corpus papers, min/median/90%/max: [ 1.  1.  4. 34.]


Q3_all | C + T: 6.0058 [5.6744, 6.3565]


Q3_ind | C + T: 3.2967 [3.0752, 3.5343]


...

Q3_ind | C + T + log1p(prior papers): 3.1023 [2.8986, 3.3202]


Q3_all | C + T + journal + year: 2.2653 [2.1306, 2.4085]


Q3_ind | C + T + journal + year: 1.1826 [1.0981, 1.2736]


Q3_all | C + T + journal + year + log1p(prior papers): 2.2933 [2.1640, 2.4305]


Q3_ind | C + T + journal + year + log1p(prior papers): 1.2062 [1.1221, 1.2965]


## Summary

| number | value | status |
|---|---|---|
| crude rate ratio, full table | 6.32, CI 6.04 to 6.59 | crude association, author cluster bootstrap |
| seeded entries | 1,784, of which 756 rides and 1,028 independent | confirmed by independent Python and Prolog implementations |
| yearly contrast | 5.46 to 6.93 from 2021 | descriptive, no stability claim |
| T coverage | 17.2% of rows, 99.0% of seeded entries | Pierre's official v5 export |
| Q3_ind, crude | 3.64, CI 3.42 to 3.85 | headline form documented in the August entry, author cluster bootstrap |
| Q3_ind, topic-adjusted | 3.34, CI 3.12 to 3.58 | headline, measurable-T rows, author-clustered delta-method interval |
| crude ratio, complete cases | 10.16 | shows the population shift |
| topic-adjusted ratio, Q3_all | 6.09, CI 5.76 to 6.45 | measurable-T rows, author-clustered delta-method interval |
| C + T, matched rows | Q3_all 6.01 [5.67, 6.36], Q3_ind 3.30 [3.08, 3.53] | same 1,088,420 rows as the sensitivities |
| with journal and year effects | Q3_all 2.27 [2.13, 2.41], Q3_ind 1.18 [1.10, 1.27] | one zero-event journal excluded for both outcomes |
| C + T + log1p(prior papers) | Q3_all 5.62 [5.33, 5.93], Q3_ind 3.10 [2.90, 3.32] | same matched rows |
| also with journal and year | Q3_all 2.29 [2.16, 2.43], Q3_ind 1.21 [1.12, 1.30] | same matched rows; nominal author-clustered delta intervals |

Removing the whole-period threshold changed the adjusted ratio from 5.24 to 6.09. On the shared rows, replacing the old topic values with the new ones changes it from 5.24 to 5.69. Adding the newly covered rows changes it further to 6.09. This sequence is descriptive and depends on the order of the steps. In the complete cases, `T` has a positive adjusted association with entry (OR 2.41 per 0.1 increase in topic fit). Step 9 shows how much the adjusted ratio depends on the specification: with journal and year effects it falls from 6.01 to 2.27 for Q3_all and from 3.30 to 1.18 for Q3_ind on the same rows. That is a comparison of two model-based associations, not a decomposition of the crude ratio into causes. The September reporting position for Kevin’s part, proposed for the joint report, keeps the form documented in the August entry, Q3_ind adjusted for T, as the headline and reports the journal and year result next to it in full, so 6.09 or 3.34 is never shown without 2.27 or 1.18 beside it.

The prior-paper sensitivity lowers the matched C+T ratios from 6.01 to 5.62 and from 3.30 to 3.10. With journal/year terms they instead rise slightly, from 2.27 to 2.29 and from 1.18 to 1.21. This does not support treating prior paper count as a simple explanation that removes the association. The intervals hold the included covariates fixed, assume independent author clusters and do not cover shared-paper dependence, selection or specification choice.

Q3_ind is the headline form documented in the August entry (decisions log, 2026-08-08): entries without the seed co-author on the entering paper, against the unseeded entry rate. It is lower than Q3_all by construction, the rides leave the numerator, and its point estimate stays above one in every reported specification, with a much smaller ratio once journal and year are held constant. So seeded authors enter more often even when the seed co-author is not on the paper, and how much of that remains after journal and year is the sensitivity result.

Implemented scope and reporting position (2026-09-09; team adoption proposed)

1. Complete-case fitting is retained. Observed T remains continuous and missing T is not zero-coded or imputed. The results describe the measurable-T population only; coverage and missingness remain limitations.
2. Q3_ind is the primary outcome. The original C + T comparison is reported with the journal/year sensitivity beside it, with the matched-population contrast and exclusion stated. The conclusion is specification-sensitive association, not identified causation. See D_results/README.md for the reporting position, proposed team adoption and deviations from the earlier plan.
